# Fink/LSST — Reload and Display Saved Light Curves

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_02"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per group
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ────────────────────────────────────────────────────────
# Choose which groups to display:
#   'all'     -> all groups found on disk (default)
#   'calib'   -> only groups usable for photometric calibration
#   'exclude' -> only groups EXCLUDED from calibration
#                (variable stars, SSO objects, TNS transients, blazars...)
PLOT_MODE = "all"  # <── change here: 'all' | 'calib' | 'exclude'

# Groups excluded from calibration (consistent with notebook 01).
# Any group whose name STARTS WITH 'simbad_variable' is also excluded.
CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "gaia_star_variable",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (identical to notebook 01 + luptitude support)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties.

    Negative or zero flux values are silently returned as NaN.

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.
    flux_err_nJy : array-like or None
        1-sigma flux uncertainty in nJy.  If None, only magnitudes are returned.

    Returns
    -------
    mag : ndarray
        AB magnitude (NaN where flux <= 0).
    mag_err : ndarray or None
        Magnitude uncertainty (NaN where flux <= 0), or None if not requested.
    """
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes).

    Luptitudes (Lupton et al. 1999) handle zero and negative flux values
    that arise in difference-image photometry (DIA).  They behave as
    standard AB magnitudes at high signal-to-noise and transition to a
    linear flux scale near the noise floor, so every measurement is
    plotted -- including those with negative flux.

    The softening parameter *b* is set to the median flux uncertainty of
    the input array, placing the linear/log transition at the 1-sigma
    noise level.  This matches the convention used in the DP2 DiaObject
    Butler notebook (``plot_dia_object_luptitudes``).

    Formula
    -------
    mu = -2.5 * log10(e) * [arcsinh(f / 2b) + ln(b)] + (zp + 2.5*log10(b))

    Error propagation
    -----------------
    sigma_mu = 1.085736 * sigma_f / sqrt(f^2 + (2b)^2)

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.  May contain negative values.
    flux_err_nJy : array-like
        1-sigma flux uncertainty in nJy (used to derive the softening
        parameter b = median(flux_err_nJy)).
    zero_point : float, optional
        AB zero-point offset in nJy units (default 31.4 for nJy).

    Returns
    -------
    lup : ndarray
        Asinh magnitude (Luptitude) for each measurement.
    lup_err : ndarray
        1-sigma uncertainty on the Luptitude.
    b : float
        Softening parameter used (= median of flux_err_nJy).
    """
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)

    # Softening parameter: median noise level of the input data
    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0  # safe fallback

    # Pre-factor  2.5 / ln(10) ~ 1.085736
    log10_e_inv = 1.085736

    # Asinh magnitude (Luptitude)
    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))

    # Error propagation
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)

    return lup, lup_err, b


print("flux_to_luptitude() defined.")

## 3. Load Parquet files from disk

In [ ]:
# Auto-discover available groups from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))


# ── Apply PLOT_MODE filter ───────────────────────────────────────────────
def _is_excluded(g):
    return g in CALIBRATION_EXCLUDE or g.startswith("simbad_variable")


if PLOT_MODE == "calib":
    selected_groups = [g for g in all_groups if not _is_excluded(g)]
elif PLOT_MODE == "exclude":
    selected_groups = [g for g in all_groups if _is_excluded(g)]
else:  # 'all'
    selected_groups = list(all_groups)

print(f"Groups on disk: {len(all_groups)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_groups)}")
for g in all_groups:
    has_fp = "yes" if g in groups_fp else "no"
    has_src = "yes" if g in groups_src else "no"
    sel = "<<" if g in selected_groups else "  "
    label = "[EXCL] " if _is_excluded(g) else "[calib]"
    print(f"  {sel} {label} {g:40s}  fp={has_fp:3s}  src={has_src}")

In [ ]:
# Load all Parquet files and reconstruct the lc_cache dictionary.
# Structure: lc_cache[group][oid] = {'fp': DataFrame, 'src': DataFrame}

lc_cache = {}

for group in all_groups:
    lc_cache[group] = {}

    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])

        # Drop any residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag / mag_err if absent (e.g. columns were not saved)
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # Split by object
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

# Summary
print("Objects loaded per group:")
for g in all_groups:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:35s}  {n_oids:3d} objects  {n_pts:6d} data points")

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['fp']

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['src']

## 4. Load flatness metrics

In [ ]:
csv_path = os.path.join(DIR_DATA, "flatness_metrics.csv")
if os.path.exists(csv_path):
    df_flat = pd.read_csv(csv_path)
    print(f"flatness_metrics.csv loaded: {len(df_flat)} rows")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
else:
    df_flat = pd.DataFrame()
    print("flatness_metrics.csv not found — flatness plots will be skipped.")

## 5. Flatness boxplot per group

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by source class — lower = flatter", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_group")
    plt.show()

## 6. Light-curve plotting functions

In [ ]:
def rank_oids(group, nc=NC):
    """Return the top-nc object IDs for a group, ranked by total number of data points."""
    scored = [(oid, len(d["fp"]) + len(d["src"])) for oid, d in lc_cache[group].items()]
    return [oid for oid, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def getTminTmax(df, df_src, df_fp):
    """ """

    # find tmin, tmax on all points / all bands
    t = df["r:midpointMjdTai"].values
    t_src = df_src["r:midpointMjdTai"].values
    t_fp = df_fp["r:midpointMjdTai"].values

    # find the first point
    if len(t_src) > 0:
        tmin = t_src.min()
    else:
        tmin = t.min()
    tmax = t.max()
    return tmin, tmax


def plot_lc_grid(group, mode="flux", nc=NC):
    """
    Plot a (n_objects x n_bands) grid of light curves for one source group.

    Parameters
    ----------
    group : str
        Group name (key of lc_cache).
    mode : str
        Display mode:
        - 'flux'      : raw PSF flux in nJy.
        - 'mag'       : AB magnitude (NaN for non-positive flux).
        - 'luptitude' : asinh magnitude, handles zero and negative
                        differential flux gracefully (recommended for DIA).
    nc : int
        Maximum number of objects to plot.

    Notes on 'luptitude' mode
    -------------------------
    Luptitudes (Lupton et al. 1999) replace the logarithm with an inverse
    hyperbolic sine so that the scale is logarithmic (magnitude-like) at
    high flux and linear near zero.  Every measurement is plotted,
    including those with negative flux arising from DIA noise fluctuations.
    The softening parameter b is set per-band to the median flux
    uncertainty, following the convention of the DP2 DiaObject Butler
    notebook (plot_dia_object_luptitudes).
    """

    STYLE = {
        "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),  # ▽ triangle inversé vide
        "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),  # ● disque plein
    }

    top = rank_oids(group, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for group {group}.")
        return

    fig, axes = plt.subplots(
        n_obj,
        len(BANDS) + 1,
        figsize=(2.8 * (len(BANDS) + 1), 2.6 * n_obj),
        sharex=False,
        sharey=False,
        squeeze=False,
    )

    # Loop on objects
    for row_idx, oid in enumerate(top):
        # Retrieve forced-photometry and detection-based light curves
        d = lc_cache[group][oid]
        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        # keep concat
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        # Defensive NaN removal on the concatenated light curve
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )
        # also clean fp and src separately
        df_fp = d["fp"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_fp = df_fp.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
        df_src = d["src"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_src = df_src.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        # find tmin, tmax on all points / all bands
        t = df_all["r:midpointMjdTai"].values
        t_src = df_src["r:midpointMjdTai"].values
        t_fp = df_fp["r:midpointMjdTai"].values

        # find the first point
        # if len(t_src) > 0:
        #    tmin = t_src.min()
        # else:
        #    tmin = t.min()
        # tmax = t.max()
        tmin, tmax = getTminTmax(df_all, df_src, df_fp)

        dtmin = t.min() - tmin
        dtmax = tmax - tmin

        first_band = -1
        ax_first_band = None
        ax_last_allbands = axes[row_idx][len(BANDS)]

        # Loop on photometric bands
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")
            df_b_src = df_src[df_src["r:band"] == band].sort_values("r:midpointMjdTai")
            df_b_fp = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")

            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            if first_band == -1:
                first_band = col_idx
                ax_first_band = ax

            # Select y-values according to the requested display mode
            if mode == "flux":
                # Finite mask on raw flux
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
                df_b = df_b[mask].reset_index(drop=True)

                mask_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(
                    df_b_src["r:psfFluxErr"].values
                )
                df_b_src = df_b_src[mask_src].reset_index(drop=True)

                mask_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(
                    df_b_fp["r:psfFluxErr"].values
                )
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                y = df_b["r:psfFlux"].values
                yerr = df_b["r:psfFluxErr"].values

                y_src = df_b_src["r:psfFlux"].values
                yerr_src = df_b_src["r:psfFluxErr"].values

                y_fp = df_b_fp["r:psfFlux"].values
                yerr_fp = df_b_fp["r:psfFluxErr"].values

            elif mode == "mag":
                # Standard AB magnitude -- NaN for non-positive flux
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
                df_b = df_b[mask].reset_index(drop=True)

                mask_src = np.isfinite(df_b_src["mag"].values) & np.isfinite(df_b_src["mag_err"].values)
                df_b_src = df_b_src[mask_src].reset_index(drop=True)

                mask_fp = np.isfinite(df_b_fp["mag"].values) & np.isfinite(df_b_fp["mag_err"].values)
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                y = df_b["mag"].values
                yerr = df_b["mag_err"].values

                y_src = df_b_src["mag"].values
                yerr_src = df_b_src["mag_err"].values

                y_fp = df_b_fp["mag"].values
                yerr_fp = df_b_fp["mag_err"].values

                ax.invert_yaxis()

            elif mode == "luptitude":
                # Asinh magnitude: handles zero and negative flux values.
                # All finite measurements are kept (no flux > 0 requirement).
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
                df_b = df_b[mask].reset_index(drop=True)

                mask_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(
                    df_b_src["r:psfFluxErr"].values
                )
                df_b_src = df_b_src[mask_src].reset_index(drop=True)

                mask_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(
                    df_b_fp["r:psfFluxErr"].values
                )
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                y, yerr, b_soft = flux_to_luptitude(df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values)
                y_src, yerr_src, b_soft_src = flux_to_luptitude(
                    df_b_src["r:psfFlux"].values, df_b_src["r:psfFluxErr"].values
                )
                y_fp, yerr_fp, b_soft_fp = flux_to_luptitude(
                    df_b_fp["r:psfFlux"].values, df_b_fp["r:psfFluxErr"].values
                )

                ax.invert_yaxis()  # brighter = lower numeric value, same as magnitudes

            else:
                raise ValueError(f"Unknown mode {mode!r}. Choose 'flux', 'mag', or 'luptitude'.")

            t = df_b["r:midpointMjdTai"].values
            t_src = df_b_src["r:midpointMjdTai"].values
            t_fp = df_b_fp["r:midpointMjdTai"].values

            dt = t - tmin
            dt_src = t_src - tmin
            dt_fp = t_fp - tmin

            color = BAND_COLORS.get(band, "gray")

            # ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)
            ax.errorbar(dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"])
            ax.errorbar(dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"])

            ax_last_allbands.errorbar(
                dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"]
            )
            ax_last_allbands.errorbar(
                dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"]
            )

            rms = rms_variability(df_b["r:psfFlux"].values)

            # Sub-title: show softening parameter for luptitudes
            if mode == "luptitude":
                ax.set_title(f"{band} N={len(df_b)} b={b_soft:.1f}nJy", fontsize=7, pad=2, color=color)
            else:
                ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)

            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)
            ax.set_xlim(dtmin - 2, dtmax + 2)

            # end loop on bands

        # work now on axis for the current object
        # Y-axis label: object ID + DDF field + display unit
        _ylabel_map = {
            "flux": "flux (nJy)",
            "mag": "AB mag",
            "luptitude": "asinh mag (luptitude)",
        }
        ylabel_unit = _ylabel_map.get(mode, mode)

        if ax_first_band is not None:
            deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
            field_label = f"  [{deep_field}]" if deep_field else ""
            ax_first_band.set_ylabel(f"{oid}{field_label}\n{ylabel_unit}", fontsize=10)

        else:
            axes[row_idx][0].set_ylabel(f"{oid}\n{ylabel_unit}", fontsize=8)

        deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
        ax_last_allbands.set_xlabel("Δt (days)", fontsize=7)
        ax_last_allbands.tick_params(labelsize=7)
        ax_last_allbands.set_ylabel(f"{ylabel_unit}", fontsize=10)
        ax_last_allbands.set_title(f"{oid} ({deep_field})", fontsize=8)
        ax_last_allbands.set_xlim(dtmin - 2, dtmax + 2)

        # Invert y-axis of the all-bands panel ONCE after the band loop.
        # Calling invert_yaxis() inside the loop toggles it on every band
        # iteration, leaving the axis un-inverted when the count is even.
        if mode in ("mag", "luptitude"):
            ax_last_allbands.invert_yaxis()

        # end of loops on objects

    fig.suptitle(f"Group: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plotting functions defined (modes: flux / mag / luptitude).")

## 7. Light curves in flux (nJy)

Controlled by `PLOT_MODE` (section 1): `'all'` · `'calib'` · `'exclude'`.

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['fp']['field'].unique().item()

In [ ]:
# lc_cache is loaded entirely (all groups).
# groups_to_plot is filtered here by PLOT_MODE via selected_groups.
groups_to_plot = [g for g in selected_groups if g in lc_cache and len(lc_cache[g]) >= 1]

for group in groups_to_plot:
    n_obj = len(lc_cache[group])
    print(f"\n=== {group} ({n_obj} objects) — flux ===")
    plot_lc_grid(group, mode="flux")

## 8. Light curves in AB magnitude

Same `PLOT_MODE` selection as section 7.  Measurements with non-positive flux are
dropped (NaN).  Use section 9 (luptitudes) to visualise all data points.

In [ ]:
for group in groups_to_plot:
    n_obj = len(lc_cache[group])
    print(f"\n=== {group} ({n_obj} objects) — magnitude ===")
    plot_lc_grid(group, mode="mag")

## 9. Light curves as Luptitudes (asinh magnitudes)

Luptitudes handle zero and negative PSF flux values that routinely appear
in difference-image photometry (DIA).  The conversion follows Lupton et al.
(1999) with a per-band softening parameter *b* equal to the median flux
uncertainty.  Every measurement is plotted -- there are no missing points
due to non-positive flux -- making it easier to judge whether variability is
real or just noise fluctuation around zero.

Same `PLOT_MODE` selection as sections 7--8.

In [ ]:
# Luptitude light curves -- controlled by PLOT_MODE (section 1)
# Every data point is plotted (no NaN due to non-positive flux).
for group in groups_to_plot:
    n_obj = len(lc_cache[group])
    print(f"\n=== {group} ({n_obj} objects) — luptitude ===")
    plot_lc_grid(group, mode="luptitude")

## 10. Detailed view — single object inspector (flux + magnitude + luptitude)

Set `TARGET_GROUP` and `TARGET_OID` below to inspect any individual object.
The detailed plot now shows **three panels** per band:

1. **Flux (nJy)** — raw PSF flux with zero-flux reference line.
2. **AB magnitude** — standard log scale (NaN for non-positive flux).
3. **Luptitude** — asinh magnitude that includes all measurements.

In [ ]:
# ── Choose the group and object to inspect ────────────────────────────────────
TARGET_GROUP = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(TARGET_GROUP, nc=5)
TARGET_OID = TOP_OIDS[0]  # ← change here (pick from TOP_OIDS)

print(f"Group    : {TARGET_GROUP}")
print(f"Object   : {TARGET_OID}")
print(f"Top OIDs : {TOP_OIDS}")

# ── Build combined light curve ────────────────────────────────────────────────
d = lc_cache[TARGET_GROUP][TARGET_OID]
frames = [df for df in (d["fp"], d["src"]) if not df.empty]
df_obj = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)
df_obj_src = (
    d["src"]
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)

df_obj_fp = (
    d["fp"]
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)

tmin, tmax = getTminTmax(df_obj, df_obj_src, df_obj_fp)
print(f"\nData points per band:")
print(df_obj.groupby("r:band").size().rename("n_pts").to_frame())

In [ ]:
STYLE = {
    "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),  # ▽ triangle inversé vide
    "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),  # ● disque plein
}

# Detailed flux + magnitude + luptitude three-panel plot per band
bands_obj = [b for b in BANDS if b in df_obj["r:band"].values]

fig, axes = plt.subplots(len(bands_obj) + 1, 3, figsize=(16, 3.2 * (len(bands_obj) + 1)), squeeze=False)

# Loop on bands
for row_idx, band in enumerate(bands_obj):
    df_b = df_obj[df_obj["r:band"] == band].copy()
    df_b_src = df_obj_src[df_obj_src["r:band"] == band].copy()
    df_b_fp = df_obj_fp[df_obj_fp["r:band"] == band].copy()

    t = df_b["r:midpointMjdTai"].values
    t_src = df_b_src["r:midpointMjdTai"].values
    t_fp = df_b_fp["r:midpointMjdTai"].values

    dt = t - tmin
    dt_src = t_src - tmin
    dt_fp = t_fp - tmin
    dtmin = dt.min()
    dtmax = dt.max()

    color = BAND_COLORS.get(band, "gray")

    # --- Panel 0: PSF flux (nJy) ---
    ax0 = axes[row_idx][0]
    ax0_last = axes[len(bands_obj)][0]

    mask_f = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    mask_f_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(df_b_src["r:psfFluxErr"].values)
    mask_f_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(df_b_fp["r:psfFluxErr"].values)

    ax0.errorbar(
        dt_src[mask_f_src],
        df_b_src["r:psfFlux"].values[mask_f_src],
        yerr=df_b_src["r:psfFluxErr"].values[mask_f_src],
        lw=1.0,
        elinewidth=1.2,
        color=color,
        **STYLE["src"],
    )
    ax0.errorbar(
        dt_fp[mask_f_fp],
        df_b_fp["r:psfFlux"].values[mask_f_fp],
        yerr=df_b_fp["r:psfFluxErr"].values[mask_f_fp],
        lw=0,
        elinewidth=0.5,
        color=color,
        **STYLE["fp"],
    )
    ax0_last.errorbar(
        dt_src[mask_f_src],
        df_b_src["r:psfFlux"].values[mask_f_src],
        yerr=df_b_src["r:psfFluxErr"].values[mask_f_src],
        lw=1.0,
        elinewidth=1.2,
        color=color,
        **STYLE["src"],
    )
    ax0_last.errorbar(
        dt_fp[mask_f_fp],
        df_b_fp["r:psfFlux"].values[mask_f_fp],
        yerr=df_b_fp["r:psfFluxErr"].values[mask_f_fp],
        lw=0,
        elinewidth=0.5,
        color=color,
        **STYLE["fp"],
    )

    rms = rms_variability(df_b["r:psfFlux"].values[mask_f])
    ax0.axhline(0.0, color="black", lw=0.6, ls="--", alpha=0.4)  # zero-flux reference
    ax0.set_xlim(dtmin - 2, dtmax + 2)
    ax0.set_ylabel(f"Flux (nJy) — band {band}", color=color)
    ax0.set_xlabel("Δt (days)")
    ax0.set_title(f"Flux  N={mask_f.sum()}  σ/<f>={rms:.4f}", fontsize=8, color=color)

    ax0_last.set_xlim(dtmin - 2, dtmax + 2)
    ax0_last.set_xlabel("Δt (days)")
    ax0_last.set_ylabel("Flux (nJy)")
    ax0_last.set_title(f"Fluxes for {TARGET_OID} ({TARGET_GROUP})")

    # --- Panel 1: AB magnitude (NaN for non-positive flux) ---
    ax1 = axes[row_idx][1]
    ax1.invert_yaxis()  # astronomical convention: brighter = top
    ax1_last = axes[len(bands_obj)][1]
    # ax1_last.invert_yaxis() moved after band loop (see below)

    if "mag" in df_b.columns and "mag_err" in df_b.columns:
        mask_m = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
        mask_m_src = np.isfinite(df_b_src["mag"].values) & np.isfinite(df_b_src["mag_err"].values)
        mask_m_fp = np.isfinite(df_b_fp["mag"].values) & np.isfinite(df_b_fp["mag_err"].values)

        ax1.errorbar(
            dt_src[mask_m_src],
            df_b_src["mag"].values[mask_m_src],
            yerr=df_b_src["mag_err"].values[mask_m_src],
            lw=1.0,
            elinewidth=1.2,
            color=color,
            **STYLE["src"],
        )
        ax1.errorbar(
            dt_fp[mask_m_fp],
            df_b_fp["mag"].values[mask_m_fp],
            yerr=df_b_fp["mag_err"].values[mask_m_fp],
            lw=0.0,
            elinewidth=0.5,
            color=color,
            **STYLE["fp"],
        )
        ax1_last.errorbar(
            dt_src[mask_m_src],
            df_b_src["mag"].values[mask_m_src],
            yerr=df_b_src["mag_err"].values[mask_m_src],
            lw=1.0,
            elinewidth=1.2,
            color=color,
            **STYLE["src"],
        )
        ax1_last.errorbar(
            dt_fp[mask_m_fp],
            df_b_fp["mag"].values[mask_m_fp],
            yerr=df_b_fp["mag_err"].values[mask_m_fp],
            lw=0.0,
            elinewidth=0.5,
            color=color,
            **STYLE["fp"],
        )

        ax1.set_xlim(dtmin - 2, dtmax + 2)
        ax1.set_ylabel(f"AB mag — band {band}", color=color)
        ax1.set_xlabel("Δt (days)")
        ax1.set_title(f"AB mag  N={mask_m.sum()} (neg. flux dropped)", fontsize=8, color=color)

        ax1_last.set_xlim(dtmin - 2, dtmax + 2)
        ax1_last.set_xlabel("Δt (days)")
        ax1_last.set_ylabel("AB mag")
        ax1_last.set_title(f"Magnitudes for {TARGET_OID} ({TARGET_GROUP})")

    else:
        ax1.set_visible(False)

    # --- Panel 2: Luptitude / asinh magnitude ---
    # All finite measurements are shown, including those with negative flux.
    # This is the recommended representation for DIA differential fluxes.
    ax2 = axes[row_idx][2]
    ax2.invert_yaxis()  # brighter = top, same convention as magnitudes
    ax2_last = axes[len(bands_obj)][2]
    # ax2_last.invert_yaxis() moved after band loop (see below)

    mask_l = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    mask_l_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(df_b_src["r:psfFluxErr"].values)
    mask_l_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(df_b_fp["r:psfFluxErr"].values)

    if mask_l.sum() >= 3:
        lup, lup_err, b_soft = flux_to_luptitude(
            df_b["r:psfFlux"].values[mask_l],
            df_b["r:psfFluxErr"].values[mask_l],
        )
        lup_src, lup_err_src, b_soft_src = flux_to_luptitude(
            df_b_src["r:psfFlux"].values[mask_l_src],
            df_b_src["r:psfFluxErr"].values[mask_l_src],
        )
        lup_fp, lup_err_fp, b_soft_fp = flux_to_luptitude(
            df_b_fp["r:psfFlux"].values[mask_l_fp],
            df_b_fp["r:psfFluxErr"].values[mask_l_fp],
        )
        ax2.errorbar(
            dt_src[mask_l_src], lup_src, yerr=lup_err_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"]
        )
        ax2.errorbar(
            dt_fp[mask_l_fp], lup_fp, yerr=lup_err_fp, lw=0.0, elinewidth=0.5, color=color, **STYLE["fp"]
        )
        ax2_last.errorbar(
            dt_src[mask_l_src], lup_src, yerr=lup_err_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"]
        )
        ax2_last.errorbar(
            dt_fp[mask_l_fp], lup_fp, yerr=lup_err_fp, lw=0.0, elinewidth=0.5, color=color, **STYLE["fp"]
        )

        ax2.set_xlim(dtmin - 2, dtmax + 2)
        ax2.set_ylabel(f"Luptitude — band {band}", color=color)
        ax2.set_xlabel("Δt (days)")
        ax2.set_title(f"Luptitude  N={mask_l.sum()}  b={b_soft:.1f} nJy", fontsize=8, color=color)

        ax2_last.set_xlim(dtmin - 2, dtmax + 2)
        ax2_last.set_xlabel("Δt (days)")
        ax2_last.set_ylabel("Luptitude")
        ax2_last.set_title(f"Luptitudes for {TARGET_OID}({TARGET_GROUP})")

    else:
        ax2.set_visible(False)

# Invert y-axis of the all-bands summary panels once, after the band loop.
# (Calling invert_yaxis() inside the loop toggles it each iteration;
#  an even number of bands leaves the axis un-inverted.)
axes[len(bands_obj)][1].invert_yaxis()  # magnitudes all-bands panel
axes[len(bands_obj)][2].invert_yaxis()  # luptitudes all-bands panel

fig.suptitle(
    f"diaObjectId = {TARGET_OID}  |  group = {TARGET_GROUP}",
    fontsize=11,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
savefig(f"03_detail_{TARGET_OID}")
plt.show()

In [ ]:
def plot_singleobjlightcurve(
    lc_cache,
    target_group,
    target_oid,
    bands,
    band_colors,
    modes=("flux", "mag", "luptitude"),  # panels à afficher
    show_per_band=True,
    show_combined=True,
    style=None,
    save=False,
    filename_prefix="03_detail",
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if style is None:
        style = {
            "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),
            "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),
        }

    d = lc_cache[target_group][target_oid]

    # ── Data prep ─────────────────────────────────────────────
    frames = [df for df in (d["fp"], d["src"]) if not df.empty]

    df_obj = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
        .sort_values("r:midpointMjdTai")
        .reset_index(drop=True)
    )

    def clean(df):
        return (
            df.drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
            .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
            .sort_values("r:midpointMjdTai")
            .reset_index(drop=True)
        )

    df_obj_src = clean(d["src"])
    df_obj_fp = clean(d["fp"])

    tmin, tmax = getTminTmax(df_obj, df_obj_src, df_obj_fp)

    bands_obj = [b for b in bands if b in df_obj["r:band"].values]

    # ── Layout logic ─────────────────────────────────────────
    n_rows = 0
    if show_per_band:
        n_rows += len(bands_obj)
    if show_combined:
        n_rows += 1

    n_cols = len(modes)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.2 * n_rows), squeeze=False)

    # mapping mode → colonne
    mode_to_col = {m: i for i, m in enumerate(modes)}

    current_row = 0

    # ── Core plotting function ───────────────────────────────
    def plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=False):
        t = df_b["r:midpointMjdTai"].values
        dt = t - tmin

        t_src = df_b_src["r:midpointMjdTai"].values
        dt_src = t_src - tmin

        t_fp = df_b_fp["r:midpointMjdTai"].values
        dt_fp = t_fp - tmin

        if mode == "flux":
            mask_src = np.isfinite(df_b_src["r:psfFlux"]) & np.isfinite(df_b_src["r:psfFluxErr"])
            mask_fp = np.isfinite(df_b_fp["r:psfFlux"]) & np.isfinite(df_b_fp["r:psfFluxErr"])

            ax.errorbar(
                dt_src[mask_src],
                df_b_src["r:psfFlux"][mask_src],
                yerr=df_b_src["r:psfFluxErr"][mask_src],
                color=color,
                **style["src"],
            )

            ax.errorbar(
                dt_fp[mask_fp],
                df_b_fp["r:psfFlux"][mask_fp],
                yerr=df_b_fp["r:psfFluxErr"][mask_fp],
                color=color,
                **style["fp"],
            )

            ax.axhline(0, color="black", lw=0.6, ls="--", alpha=0.4)
            ax.set_ylabel("Flux (nJy)")

        elif mode == "mag" and "mag" in df_b.columns:
            mask_src = np.isfinite(df_b_src["mag"]) & np.isfinite(df_b_src["mag_err"])
            mask_fp = np.isfinite(df_b_fp["mag"]) & np.isfinite(df_b_fp["mag_err"])

            ax.errorbar(
                dt_src[mask_src],
                df_b_src["mag"][mask_src],
                yerr=df_b_src["mag_err"][mask_src],
                color=color,
                **style["src"],
            )

            ax.errorbar(
                dt_fp[mask_fp],
                df_b_fp["mag"][mask_fp],
                yerr=df_b_fp["mag_err"][mask_fp],
                color=color,
                **style["fp"],
            )

            ax.invert_yaxis()
            ax.set_ylabel("AB mag")

        elif mode == "luptitude":
            mask_src = np.isfinite(df_b_src["r:psfFlux"]) & np.isfinite(df_b_src["r:psfFluxErr"])
            mask_fp = np.isfinite(df_b_fp["r:psfFlux"]) & np.isfinite(df_b_fp["r:psfFluxErr"])

            if mask_src.sum() >= 3:
                lup_src, lup_err_src, _ = flux_to_luptitude(
                    df_b_src["r:psfFlux"][mask_src],
                    df_b_src["r:psfFluxErr"][mask_src],
                )
                lup_fp, lup_err_fp, _ = flux_to_luptitude(
                    df_b_fp["r:psfFlux"][mask_fp],
                    df_b_fp["r:psfFluxErr"][mask_fp],
                )

                ax.errorbar(dt_src[mask_src], lup_src, yerr=lup_err_src, color=color, **style["src"])
                ax.errorbar(dt_fp[mask_fp], lup_fp, yerr=lup_err_fp, color=color, **style["fp"])

                ax.invert_yaxis()
                ax.set_ylabel("Luptitude")

        ax.set_xlabel("Δt (days)")
        if not is_combined:
            ax.set_title(f"{mode} — band {band}", fontsize=8, color=color)

    # ── Per-band plots ───────────────────────────────────────
    if show_per_band:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = df_obj_src[df_obj_src["r:band"] == band]
            df_b_fp = df_obj_fp[df_obj_fp["r:band"] == band]

            color = band_colors.get(band, "gray")

            for mode in modes:
                ax = axes[current_row][mode_to_col[mode]]
                plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color)

            current_row += 1

    # ── Combined plot ────────────────────────────────────────
    if show_combined:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = df_obj_src[df_obj_src["r:band"] == band]
            df_b_fp = df_obj_fp[df_obj_fp["r:band"] == band]

            color = band_colors.get(band, "gray")

            for mode in modes:
                ax = axes[current_row][mode_to_col[mode]]
                plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=True)

        for mode in modes:
            axes[current_row][mode_to_col[mode]].set_title(f"{mode} (all bands)")
            ax = axes[current_row][mode_to_col[mode]]
            # simple toggle on y axis
            if mode in ("mag", "luptitude"):
                ax.invert_yaxis()
            # really force the inversion
            # if mode in ("mag", "luptitude"):
            #    ax.set_ylim(ax.get_ylim()[::-1])

    # ── Final touches ────────────────────────────────────────
    fig.suptitle(f"diaObjectId = {target_oid} | group = {target_group}", fontsize=11)

    plt.tight_layout()

    if save:
        plt.savefig(f"{filename_prefix}_{target_oid}.png")

    plt.show()

In [ ]:
plot_singleobjlightcurve(
    lc_cache,
    TARGET_GROUP,
    TARGET_OID,
    BANDS,
    BAND_COLORS,
    modes=("flux", "mag", "luptitude"),
    # modes=("flux",),
    show_per_band=True,
    show_combined=True,
)

## 11. Renormalisation

In [ ]:
def choose_reference_band(df_src, df_all, bands_priority=("i", "r", "z", "g", "y"), min_pts=5):
    for b in bands_priority:
        n_src = (df_src["r:band"] == b).sum()
        if n_src >= min_pts:
            return b, "src"

    for b in bands_priority:
        n_all = (df_all["r:band"] == b).sum()
        if n_all >= min_pts:
            return b, "all"

    return None, None

## 12. Calibration summary scatter plot

Variability vs mean flux per group, per band.

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"],
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Calibration suitability  (best = bright & flat = bottom-right)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("04_calibration_summary")
    plt.show()

    # Final ranking table
    print("\nRanking by photometric flatness (ascending RMS):")
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    display(ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag"]])